In [0]:
%run ../connection

In [0]:

 
import requests   
import json       
import uuid      
import time      
import logging   
 
from datetime import datetime, timezone  
from typing   import Any, Dict, List, Optional

In [0]:
%run ./config

In [0]:

def call_api(endpoint: str, params: Optional[Dict] = None) -> Any:
    """
    Make a GET request to the CoinGecko API with retry logic.
    """
    if params is None:
        params = {}
 
    # Always inject the API key — never inline it in the URL string
    params["x_cg_demo_api_key"] = API_KEY
 
    url = f"{BASE_URL}{endpoint}"
 
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            logger.debug(f"Calling {url} | attempt {attempt}")
            response = requests.get(url, params=params, timeout=30)
 
            response.raise_for_status()
 
            return response.json()
 
        except requests.exceptions.HTTPError as e:
            status = e.response.status_code if e.response else "unknown"
 
            if status == 429:
                wait = RETRY_BACKOFF_SEC * attempt * 2
                logger.warning(
                    f"Rate limit hit (429) on {endpoint}. "
                    f"Waiting {wait}s before retry {attempt}/{MAX_RETRIES}"
                )
                time.sleep(wait)
            elif attempt == MAX_RETRIES:
                logger.error(f"All {MAX_RETRIES} retries failed for {endpoint}: {e}")
                raise
            else:
                logger.warning(f"HTTP {status} on attempt {attempt} for {endpoint}: {e}")
                time.sleep(RETRY_BACKOFF_SEC)
 
        except requests.exceptions.Timeout:
            logger.warning(f"Timeout on attempt {attempt} for {endpoint}")
            if attempt == MAX_RETRIES:
                raise
            time.sleep(RETRY_BACKOFF_SEC)
 
        except requests.exceptions.ConnectionError as e:
            logger.warning(f"Connection error on attempt {attempt} for {endpoint}: {e}")
            if attempt == MAX_RETRIES:
                raise
            time.sleep(RETRY_BACKOFF_SEC)
 
 
def build_envelope(data: Any, source_endpoint: str, extra_meta: Optional[Dict] = None) -> Dict:
    """
    Wrap the raw API payload in a standard metadata envelope.
    """
    envelope = {
        "ingestion_timestamp" : NOW_UTC.isoformat(),      
        "pipeline_run_id"     : RUN_ID,
        "source_endpoint"     : source_endpoint,
        "top_n_coins_config"  : TOP_N_COINS,    
        "schema_version"      : "1.0",                 
        "payload"             : data,
    }
 
    if extra_meta:
        envelope.update(extra_meta)
 
    return envelope
 
 
def save_to_landing(envelope: Dict, subfolder: str, filename: str) -> str:
    """
    Serialize envelope to JSON and write to ADLS landing container.
    """
    full_path = f"{ADLS_ROOT}/{subfolder}/{DATE_PATH}/{filename}"
 
    json_content = json.dumps(envelope, indent=2, default=str)
 
    dbutils.fs.put(full_path, json_content, overwrite=True)

    logger.info(f"✓ Saved → {full_path}")
    return full_path

In [0]:

#ENDPOINT 1 — /coins/markets
 
logger.info("─" * 60)
logger.info("ENDPOINT 1: /coins/markets — top market snapshot")
 
markets_data = call_api(
    "/coins/markets",
    params={
        "vs_currency" : "usd",            
        "order"       : "market_cap_desc", 
        "per_page"    : TOP_N_COINS,      
        "page"        : 1,                
        "sparkline"   : "false",         
    }
)
 
markets_path = save_to_landing(
    envelope  = build_envelope(markets_data, "coins_markets"),
    subfolder = "coins_markets",
    filename  = f"coins_markets_{TS_SUFFIX}.json"
)
 
coin_ids: List[str] = [coin["id"] for coin in markets_data]
logger.info(f"  Collected {len(coin_ids)} coins: {coin_ids[:5]} ... (truncated)")
 
 

In [0]:
#CELL 6: ENDPOINTS 2 & 3 — /coins/{id}/ohlc and /coins/{id}/market_chart

logger.info("─" * 60)
logger.info(f"ENDPOINTS 2 & 3: OHLC + Market Chart for {len(coin_ids)} coins")
 
ohlc_paths         = []
market_chart_paths = []
 
for index, coin_id in enumerate(coin_ids, start=1):
    logger.info(f"  [{index:>2}/{len(coin_ids)}] Processing coin: {coin_id}")
 
    # ── Endpoint 2: OHLC 
    try:
        ohlc_data = call_api(
            f"/coins/{coin_id}/ohlc",
            params={"vs_currency": "usd", "days": 30}
        )
        ohlc_path = save_to_landing(
            envelope  = build_envelope(
                ohlc_data,
                source_endpoint = "ohlc",
                extra_meta      = {"coin_id": coin_id}   # Tag which coin this is
            ),
            subfolder = f"ohlc/{coin_id}",
            filename  = f"ohlc_{coin_id}_{TS_SUFFIX}.json"
        )
        ohlc_paths.append(ohlc_path)
    except Exception as e:
        logger.error(f"  OHLC failed for {coin_id}: {e}")
 
    # ── Endpoint 3: Market Chart 
    try:
        chart_data = call_api(
            f"/coins/{coin_id}/market_chart",
            params={"vs_currency": "usd", "days": 30}
        )
        chart_path = save_to_landing(
            envelope  = build_envelope(
                chart_data,
                source_endpoint = "market_chart",
                extra_meta      = {"coin_id": coin_id}
            ),
            subfolder = f"market_chart/{coin_id}",
            filename  = f"market_chart_{coin_id}_{TS_SUFFIX}.json"
        )
        market_chart_paths.append(chart_path)
    except Exception as e:
        logger.error(f"  market_chart failed for {coin_id}: {e}")
 
    if index < len(coin_ids): 
        time.sleep(RATE_LIMIT_SLEEP)
 
logger.info(f"  Saved {len(ohlc_paths)} OHLC files")
logger.info(f"  Saved {len(market_chart_paths)} market_chart files")
 
 

In [0]:
# CELL 7: ENDPOINT 4 — /search/trending
 
logger.info("─" * 60)
logger.info("ENDPOINT 4: /search/trending — trending coins")
 
trending_data = call_api("/search/trending")
 
trending_path = save_to_landing(
    envelope  = build_envelope(trending_data, "trending"),
    subfolder = "trending",
    filename  = f"trending_{TS_SUFFIX}.json"
)
 
logger.info(f"  Trending coins: {[c['item']['id'] for c in trending_data.get('coins', [])]}")

In [0]:
# CELL 8: ENDPOINT 5 — /global
 
logger.info("─" * 60)
logger.info("ENDPOINT 5: /global — global market statistics")
 
global_data = call_api("/global")
 
global_path = save_to_landing(
    envelope  = build_envelope(global_data, "global"),
    subfolder = "global",
    filename  = f"global_{TS_SUFFIX}.json"
)
 
btc_dom = global_data.get("data", {}).get("market_cap_percentage", {}).get("btc", "N/A")
logger.info(f"  BTC Dominance: {btc_dom:.2f}%" if isinstance(btc_dom, float) else f"  BTC Dominance: {btc_dom}")

In [0]:

# CELL 9: INGESTION SUMMARY
summary = {
    "event"               : "landing_ingestion_complete",
    "pipeline_run_id"     : RUN_ID,
    "run_timestamp_utc"   : NOW_UTC.isoformat(),
    "coins_collected"     : len(coin_ids),
    "ohlc_files_saved"    : len(ohlc_paths),
    "market_chart_files"  : len(market_chart_paths),
    "trending_saved"      : True,
    "global_saved"        : True,
    "top_n_coins_config"  : TOP_N_COINS,
    "status"              : "SUCCESS"
}
 
logger.info("=" * 70)
logger.info("LANDING INGESTION COMPLETE")
for k, v in summary.items():
    logger.info(f"  {k:<28}: {v}")
logger.info("=" * 70)
 
 